# Task 5 — Web Data Extraction & Analysis
## Exploratory Data Analysis & Product Intelligence

**Author:** Himanshu Rana  
**Target Website:** [Books to Scrape](https://books.toscrape.com)  
**Tools:** BeautifulSoup4, Requests, Pandas, Matplotlib, Seaborn  

---
### Objective
1. Scrape product data (Titles, Prices, Ratings, Availability, Categories) from a publicly available sandbox.
2. Clean and structure the harvested information into a standardized dataset.
3. Perform exploratory data analysis to identify trends and patterns.
4. Export clean structured outputs to CSV and Excel.

## 1. Environment Setup & Data Loading
Load required libraries and import the scraped dataset from `data/processed/scraped_books.csv`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "font.sans-serif": ["DejaVu Sans", "Arial", "Segoe UI"],
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

DATA_PATH = Path("../data/processed/scraped_books.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Task_5_Web_Data_Extraction/data/processed/scraped_books.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} book records across {df['Category'].nunique()} categories.")
df.head()

## 2. Dataset Hygiene & Integrity Verification
Verify null counts, unique counts, data types, and check for duplicate records.

In [ ]:
print("--- Dataset Information ---")
print(df.info())
print("\n--- Null Value Check ---")
print(df.isna().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

## 3. Descriptive Statistics
Review key summary statistics for numerical attributes: Price (GBP) and Star Rating (1–5).

In [ ]:
df[["Price_GBP", "Rating"]].describe(percentiles=[0.25, 0.50, 0.75, 0.95]).round(2)

## 4. Price Distribution Analysis
Examine the distribution of book prices across the catalogue.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["Price_GBP"], bins=35, kde=True, color="#2980b9", ax=ax)
ax.axvline(df["Price_GBP"].median(), color="#e74c3c", ls="--", lw=2, label=f"Median: £{df['Price_GBP'].median():.2f}")
ax.axvline(df["Price_GBP"].mean(), color="#27ae60", ls=":", lw=2, label=f"Mean: £{df['Price_GBP'].mean():.2f}")
ax.set_title("Distribution of Book Prices (GBP)", fontweight="bold")
ax.set_xlabel("Price (GBP)")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Star Rating Distribution
Understand customer review ratings and how books are classified.

In [ ]:
rating_counts = df["Rating"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar([f"{int(r)} Star" for r in rating_counts.index], rating_counts.values,
              color=["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db"], edgecolor="white")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 3, f"{int(b.get_height()):,}", ha="center", fontweight="bold")
ax.set_title("Rating Distribution Across Catalog", fontweight="bold")
ax.set_xlabel("Star Rating")
ax.set_ylabel("Book Count")
plt.tight_layout()
plt.show()

## 6. Category Volume Analysis
Identify which categories contain the largest product inventory.

In [ ]:
top_cats = df["Category"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_cats.values, y=top_cats.index, palette="viridis", hue=top_cats.index, legend=False, ax=ax)
ax.set_title("Top 15 Categories by Book Count", fontweight="bold")
ax.set_xlabel("Number of Books")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

## 7. Pricing by Category & Rating Correlation
Analyze pricing variations across genre categories and check relationship with ratings.

In [ ]:
cat_price = df.groupby("Category")["Price_GBP"].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False).round(2)
print("--- Top 10 Most Expensive Categories on Average ---")
display(cat_price.head(10))

corr = df[["Price_GBP", "Rating"]].corr()
print("\n--- Correlation Matrix ---")
print(corr)

## 8. Export Summary & Verification
The data extraction pipeline automated both scraping and exports:
- Processed CSV: `data/processed/scraped_books.csv`
- Multi-sheet Excel workbook: `outputs/scraped_books.xlsx` (Books, Summary, Category Analysis)
- High-resolution Visualizations: `outputs/charts/`